In [ ]:
import os
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill
from datetime import datetime

CISA_KEV_CSV = "reference_data/known_exploited_vulnerabilities.csv"
INPUT_EXCEL = "reference_data/vulnerability_contextualization_scanner_sample_updated.xlsx"

# Coordinated Master File (Single Source of Truth)
COORDINATED_LEDGER = "reference_data/vulnerability_coordinated_ledger.xlsx"

# Dynamic timestamp headers to retain chronological histories
CURRENT_DATE = datetime.now().strftime("%Y%m%d")
AUDIT_COL = f"AUDIT_MARKING_{CURRENT_DATE}"
STATUS_COL = f"DATA_STATUS_{CURRENT_DATE}"

print(f"[+] Coordinated timeline workspace parameters loaded.")
print(f"    - Target Schema Columns: ['CVE_ID', '{AUDIT_COL}', '{STATUS_COL}', 'Data_Status (Original)']")


[+] Coordinated timeline workspace parameters loaded.
    - Target Schema Columns: ['CVE_ID', 'AUDIT_MARKING_20261030', 'DATA_STATUS_20261030', 'Data_Status (Original)']


In [39]:
if not os.path.exists(CISA_KEV_CSV) or not os.path.exists(INPUT_EXCEL):
    print("[-] Aborted: Please verify that both target database files exist inside your reference_data/ directory.")
else:
    try:
        # 1. Extract and standardize the CISA KEV reference feed
        cisa_df = pd.read_csv(CISA_KEV_CSV, encoding="utf-8-sig")
        cisa_df.columns = [str(c).strip() for c in cisa_df.columns]
        
        cisa_cve_col = next((col for col in cisa_df.columns if "CVE" in col.upper()), None)
        if cisa_cve_col:
            cisa_master_set = set(cisa_df[cisa_cve_col].dropna().astype(str).str.strip().str.upper())
            print(f"[SUCCESS] Parsed {len(cisa_master_set)} official CVE entries from local CISA KEV sheet.")
        else:
            print("[-] Error: Missing a standard CVE ID reference field in your CISA CSV document.")

        # 2. Extract the source workbook to isolate the required columns
        master_df = pd.read_excel(INPUT_EXCEL, engine="openpyxl")
        norm_cols = [str(c).strip().upper().replace(" ", "_") for c in master_df.columns]
        
        # Resolve target key column locations dynamically
        cve_idx = next((i for i, c in enumerate(norm_cols) if c in ["CVE_ID", "CVE"]), None)
        status_idx = next((i for i, c in enumerate(norm_cols) if "DATA_STATUS" in c), None)
        finding_idx = next((i for i, c in enumerate(norm_cols) if "FINDING_ID" in c), None)
        
        if cve_idx is None or status_idx is None:
            print("[-] Error: Layout normalization failed. Required 'CVE ID' or original 'Data_Status' columns are missing.")
        else:
            cve_col_name = master_df.columns[cve_idx]
            original_status_col_name = master_df.columns[status_idx]
            
            # Clean database boundaries by filtering out summary footnotes
            if finding_idx is not None:
                finding_col = master_df.columns[finding_idx]
                master_df = master_df[master_df[finding_col].notna() & master_df[finding_col].astype(str).str.contains("VULN-", na=False)]
                
            master_df = master_df[master_df[cve_col_name].notna() & (master_df[cve_col_name].astype(str).str.strip() != "")]
            master_df = master_df.reset_index(drop=True)
            print(f"[SUCCESS] Isolated {len(master_df)} clean rows ready for 4-column timeline generation.")
            
    except Exception as parse_err:
        print(f"[-] Data initialization error: {parse_err}")


[SUCCESS] Parsed 1766 official CVE entries from local CISA KEV sheet.
[SUCCESS] Isolated 300 clean rows ready for 4-column timeline generation.


In [53]:
try:
    # 1. Evaluate vector logic against CISA lookup set
    clean_cve_series = master_df[cve_col_name].astype(str).str.strip().str.upper()
    match_vector = clean_cve_series.isin(cisa_master_set)

    # 2. Populate operational marking descriptors based on the matched boolean logic
    audit_marking_values = match_vector.map({
        True: "PASS (Verified in CISA KEV Sheet)",
        False: "NOT_IN_KEV / VERIFY REQUIRED"
    })

    # 3. Generate today's new verified status logs
    updated_data_status = match_vector.map({
        True: f"VALIDATED ({CURRENT_DATE}): Record matches CISA KEV Master Ledger.",
        False: f"VALIDATED ({CURRENT_DATE}): Verified against KEV. No active exploitation logged."
    })

    # 4. Build the explicit 4-column DataFrame retaining the original Data_Status at the end
    coordinated_df = pd.DataFrame({
        "CVE_ID": clean_cve_series,
        AUDIT_COL: audit_marking_values,
        STATUS_COL: updated_data_status,
        "Data_Status": master_df[original_status_col_name]  # Pulls the unmodified scanner data status column directly
    })

    # 5. Write data out to the permanent coordinated master location
    coordinated_df.to_excel(COORDINATED_LEDGER, index=False, engine="openpyxl")

    # 6. Inject openpyxl soft orange styles to unmatched rows inside the current run column
    wb = openpyxl.load_workbook(COORDINATED_LEDGER)
    ws = wb.active
    
    # Define soft orange hex fill (Hex: FFB366)
    orange_fill = PatternFill(start_color="FFB366", end_color="FFB366", fill_type="solid")
    
    unmatched_count = 0
    for row_idx in range(2, ws.max_row + 1):
        status_cell = ws.cell(row=row_idx, column=2)  # Column 2 is your live AUDIT_MARKING_YYYYMMDD
        if status_cell.value == "NOT_IN_KEV / VERIFY REQUIRED":
            # Highlight both the CVE_ID and today's verification status field orange
            ws.cell(row=row_idx, column=1).fill = orange_fill  # Column 1 is CVE_ID
            status_cell.fill = orange_fill
            unmatched_count += 1

    wb.save(COORDINATED_LEDGER)
    print("-" * 120)
    print(f"  - Coordinated Master Workbook Location: {COORDINATED_LEDGER}")
    print(f"  - Retained Schema Fields              : ['CVE_ID', '{AUDIT_COL}', '{STATUS_COL}', 'Data_Status']")
    print(f"  - Active Mismatches Flagged (Orange)  : {unmatched_count} targets")
    print("-" * 120)

    display(coordinated_df.head(3))
    # display(coordinated_df)

except Exception as pipeline_err:
    print(f"[-] Failed to build unified coordinated ledger: {pipeline_err}")


------------------------------------------------------------------------------------------------------------------------
  - Coordinated Master Workbook Location: reference_data/vulnerability_coordinated_ledger.xlsx
  - Retained Schema Fields              : ['CVE_ID', 'AUDIT_MARKING_20261030', 'DATA_STATUS_20261030', 'Data_Status']
  - Active Mismatches Flagged (Orange)  : 89 targets
------------------------------------------------------------------------------------------------------------------------


,CVE_ID,AUDIT_MARKING_20261030,DATA_STATUS_20261030,Data_Status
0,CVE-2021-44228,PASS (Verified in CISA KEV Sheet),VALIDATED (20261030): Record matches CISA KEV ...,Synthetic scenario values; CVE/product metadat...
1,CVE-2021-45046,PASS (Verified in CISA KEV Sheet),VALIDATED (20261030): Record matches CISA KEV ...,Synthetic scenario values; CVE/product metadat...
2,CVE-2021-45105,NOT_IN_KEV / VERIFY REQUIRED,VALIDATED (20261030): Verified against KEV. No...,Synthetic scenario values; CVE/product metadat...


In [ ]:
import matplotlib.pyplot as plt

try:
    # 1. Evaluate vector logic against CISA lookup set
    clean_cve_series = master_df[cve_col_name].astype(str).str.strip().str.upper()
    match_vector = clean_cve_series.isin(cisa_master_set)

    # 2. Populate operational marking descriptors based on the matched boolean logic
    audit_marking_values = match_vector.map({
        True: "PASS (Verified in CISA KEV Sheet)",
        False: "NOT_IN_KEV / VERIFY REQUIRED"
    })

    # 3. Generate today's new verified status logs
    updated_data_status = match_vector.map({
        True: f"VALIDATED ({CURRENT_DATE}): Record matches CISA KEV Master Ledger.",
        False: f"VALIDATED ({CURRENT_DATE}): Verified against KEV. No active exploitation logged."
    })

    # 4. Build the explicit 4-column DataFrame retaining the original Data_Status at the end
    coordinated_df = pd.DataFrame({
        "CVE_ID": clean_cve_series,
        AUDIT_COL: audit_marking_values,
        STATUS_COL: updated_data_status,
        "Data_Status": master_df[original_status_col_name]  # Pulls the unmodified scanner data status column directly
    })

    # 5. Write data out to the permanent coordinated master location
    coordinated_df.to_excel(COORDINATED_LEDGER, index=False, engine="openpyxl")

    # 6. Inject openpyxl soft orange styles to unmatched rows inside the current run column
    wb = openpyxl.load_workbook(COORDINATED_LEDGER)
    ws = wb.active
    
    # Define soft orange hex fill (Hex: FFB366)
    orange_fill = PatternFill(start_color="FFB366", end_color="FFB366", fill_type="solid")
    
    unmatched_count = 0
    for row_idx in range(2, ws.max_row + 1):
        status_cell = ws.cell(row=row_idx, column=2)  # Column 2 is your live AUDIT_MARKING_YYYYMMDD
        if status_cell.value == "NOT_IN_KEV / VERIFY REQUIRED":
            # Highlight both the CVE_ID and today's verification status field orange
            ws.cell(row=row_idx, column=1).fill = orange_fill  # Column 1 is CVE_ID
            status_cell.fill = orange_fill
            unmatched_count += 1

    wb.save(COORDINATED_LEDGER)
    
    # 7. Calculate visualization variables
    total_count = len(coordinated_df)
    matched_count = total_count - unmatched_count

    # 8. Render Terminal Statistics Summary Report (Emoji-free)
    print("-" * 75)
    print("COORDINATED MASTER LEDGER INTEGRATION COMPLETE (4-COLUMN CHRONO MODE)")
    print("-" * 75)
    print(f"  - Master Ledger File Path : {COORDINATED_LEDGER}")
    print(f"  - Retained Schema Fields  : ['CVE_ID', '{AUDIT_COL}', '{STATUS_COL}', 'Data_Status']")
    print(f"  - Total Records Checked   : {total_count}")
    print(f"  - Active Matches (PASS)   : {matched_count}")
    print(f"  - Mismatches (Orange)     : {unmatched_count}")
    print("-" * 75 + "\n")

    # 9. Generate Clean Matplotlib Pie Chart Visualization
    chart_data = pd.Series([matched_count, unmatched_count], index=["PASS", "NOT_IN_KEV"])
    
    plt.figure(figsize=(5, 5))
    chart_data.plot(
        kind="pie", 
        autopct="%1.1f%%", 
        startangle=90, 
        colors=["#66BB6A", "#FFB366"],  # Soft corporate green vs the exact matching orange hex code
        textprops={"fontsize": 11, "weight": "bold"}
    )
    plt.title("CISA KEV Dataset Compliance Audit Status", fontsize=13, weight="bold", pad=20)
    plt.ylabel("")  # Removes the default pandas column label layout clutter
    plt.show()

    # Render clean 4-column table instantly below the chart in Jupyter view
    display(coordinated_df.head(3))

except Exception as pipeline_err:
    print(f"[-] Failed to build unified coordinated ledger: {pipeline_err}")


---------------------------------------------------------------------------
COORDINATED MASTER LEDGER INTEGRATION COMPLETE (4-COLUMN CHRONO MODE)
---------------------------------------------------------------------------
  - Master Ledger File Path : reference_data/vulnerability_coordinated_ledger.xlsx
  - Active Schema Columns   : ['CVE_ID', 'AUDIT_MARKING_20261030', 'DATA_STATUS_20261030', 'Data_Status']
---------------------------------------------------------------------------
📊 COMPLIANCE METRICS VISUALIZATION
---------------------------------------------------------------------------
  [PASS] Verified in CISA KEV : [#####################.........] 70.3% (211/300)
  [FAIL] Not Found / Verify   : [########......................] 29.7% (89/300)
---------------------------------------------------------------------------



,CVE_ID,AUDIT_MARKING_20261030,DATA_STATUS_20261030,Data_Status
0,CVE-2021-44228,PASS (Verified in CISA KEV Sheet),VALIDATED (20261030): Record matches CISA KEV ...,Synthetic scenario values; CVE/product metadat...
1,CVE-2021-45046,PASS (Verified in CISA KEV Sheet),VALIDATED (20261030): Record matches CISA KEV ...,Synthetic scenario values; CVE/product metadat...
2,CVE-2021-45105,NOT_IN_KEV / VERIFY REQUIRED,VALIDATED (20261030): Verified against KEV. No...,Synthetic scenario values; CVE/product metadat...
